# Continual Training (B)

Goal: deepen the model of language knowledge, teach her the grammar, syntax and semantics of the Bashkir language.

There will be 2 experiments:
- **A** - use model after vocab extension, with heated embeddings, but without alignment. Goal is to understand how basic model learns new language
- **B** - use model after embedding alignment. Goal is to checck whether that alignment give some improvement

 How to compare? - Using loss/perplexity/accuracy on task-specific dataset


## Experiment

In [1]:
EXPERIMENT = "B"   # "A" or "B"

# artifact into wandb
if EXPERIMENT == "A":
    ARTIFACT_NAME = "llama2_bashkir_vocab"                 # result of vocab extension
    WANDB_PROJECT = "bashllama-continual-training-A"
    WANDB_RUN_NAME = "continual_A"
    CHECKPOINT_ARTIFACT_PREFIX = "continual_A_checkpoint"
else:
    ARTIFACT_NAME = "llama2_bashkir_vocab_aligned"         # result of embedding alignment
    WANDB_PROJECT = "bashllama-continual-training-B"
    WANDB_RUN_NAME = "continual_B"
    CHECKPOINT_ARTIFACT_PREFIX = "continual_B_checkpoint"

print(f"Experiment {EXPERIMENT}: using artifact '{ARTIFACT_NAME}', logging to project '{WANDB_PROJECT}'")

Experiment B: using artifact 'llama2_bashkir_vocab_aligned', logging to project 'bashllama-continual-training-B'


## Imports

In [2]:
!pip install -q wandb datasets transformers accelerate bitsandbytes peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 109.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 

In [3]:
import os
import torch
import wandb
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, Trainer, TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import load_dataset, Dataset
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm import tqdm
from transformers import TrainerCallback


from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from peft import PeftModel

In [4]:
import wandb

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()


wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: e278979 (e278979-metu-middle-east-technical-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
from huggingface_hub import login, whoami
from kaggle_secrets import UserSecretsClient

login(token=UserSecretsClient().get_secret("HF_TOKEN_METU"))

In [6]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memory:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")

CUDA available: True
GPU: Tesla T4
Memory: 15.637086208 GB


## Model

Get corresponding artefact from wandb, set up 4-bit quantization and load the model


In [7]:
# base model after vocab ext
run_base = wandb.init(entity="e278979-metu-middle-east-technical-university", project="bashllama-vocab-extension")
artifact_base = run_base.use_artifact("e278979-metu-middle-east-technical-university/bashllama-vocab-extension/llama2_bashkir_vocab:v1", type="model")
base_model_dir = artifact_base.download()

wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260528_170240-vsrlfq8w
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run gallant-universe-29
wandb: ⭐️ View project at https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-vocab-extension
wandb: 🚀 View run at https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-vocab-extension/runs/vsrlfq8w
wandb: Downloading large artifact 'llama2_bashkir_vocab:v1', 4142.38MB. 6 files...
wandb:   6 of 6 files downloaded.  
Done. 00:00:27.8 (149.1MB/s)


In [8]:
# QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [9]:
model = AutoModelForCausalLM.from_pretrained(
    base_model_dir, #base model
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
    use_cache=False
)
tokenizer = AutoTokenizer.from_pretrained(base_model_dir)
tokenizer.pad_token = tokenizer.eos_token

model = prepare_model_for_kbit_training(model)

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

### Get model after embed alignment

Since it is not model itself, but peftmodel

In [10]:
run_aligned = wandb.init(entity="e278979-metu-middle-east-technical-university", project="bashllama-embedding-alignment")
artifact_aligned = run_aligned.use_artifact("e278979-metu-middle-east-technical-university/bashllama-embedding-alignment/llama2_bashkir_vocab_aligned:v0", type="model")
aligned_dir = artifact_aligned.download()

model = PeftModel.from_pretrained(model, aligned_dir)

wandb: Finishing previous runs because reinit is set to 'default'.
wandb: updating run metadata
wandb: 🚀 View run gallant-universe-29 at: https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-vocab-extension/runs/vsrlfq8w
wandb: ⭐️ View project at: https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-vocab-extension
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260528_170240-vsrlfq8w/logs
wandb: setting up run cr0bcypr
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260528_170323-cr0bcypr
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run astral-mountain-14
wandb: ⭐️ View project at https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-embedding-alignment
wandb: 🚀 View run at https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-embedding-alignment/runs/cr0bcyp

## LoRA

Add LoRA adapters. We don't specify `modules_to_save` because Continual Training should only train low-ranking adapters, not full embeddings. The embeddings have already been “warmed up” in `vocab_extension`. This saves memory and prevents overfitting.


Merge model to not have 2 lora layers

In [11]:
model = model.merge_and_unload()

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [12]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],   # as in LlamaTurk
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,194,304 || all params: 6,975,090,688 || trainable%: 0.0601


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


## Data

Get only non shuffled text from bashkort raw data

- *use non shuffled texts* - leave only high-quality, syntactically correct Bashkir language.

- *shuffle texts(order) after* -  provide a random order of presentation of examples for more stable and effective learning.

In [13]:
dataset = load_dataset("metuKKhud/bashqort-raw")

README.md: 0.00B [00:00, ?B/s]

data/bash_news_articles-00000-of-00001.p(…):   0%|          | 0.00/38.5M [00:00<?, ?B/s]

data/bashgazet_articles-00000-of-00001.p(…):   0%|          | 0.00/1.46M [00:00<?, ?B/s]

data/neftcity_articles-00000-of-00001.pa(…):   0%|          | 0.00/451k [00:00<?, ?B/s]

data/public_domain-00000-of-00001.parque(…):   0%|          | 0.00/66.6k [00:00<?, ?B/s]

...

Generating texts_jeshlek split:   0%|          | 0/6259 [00:00<?, ? examples/s]

Generating texts_kiskeufa split:   0%|          | 0/45 [00:00<?, ? examples/s]

Generating texts_kulturarb split:   0%|          | 0/1557 [00:00<?, ? examples/s]

Generating texts_president_rb split:   0%|          | 0/1879 [00:00<?, ? examples/s]

Generating texts_tabin split:   0%|          | 0/539 [00:00<?, ? examples/s]

In [14]:
clean_texts = []
for split_name, split_dataset in dataset.items():
    clean_split = split_dataset.filter(lambda x: x['is_shuffled'] is False)
    if len(clean_split) == 0:
        continue
    texts_clean = [text.replace('\n', ' ') for text in clean_split['text']]
    clean_texts.extend(texts_clean)
    print(f"{split_name}: {len(texts_clean)} texts")

print(f"Overall non shuffled: {len(clean_texts)}")

Filter:   0%|          | 0/61932 [00:00<?, ? examples/s]

bash_news_articles: 61932 texts


Filter:   0%|          | 0/803 [00:00<?, ? examples/s]

bashgazet_articles: 803 texts


Filter:   0%|          | 0/530 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/45 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1557 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1879 [00:00<?, ? examples/s]

Filter:   0%|          | 0/539 [00:00<?, ? examples/s]

Overall non shuffled: 63270


In [15]:
# create dataset and shuffle order
train_dataset = Dataset.from_dict({"text": clean_texts})
train_dataset = train_dataset.shuffle(seed=42)

In [16]:
# tokenization

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256, padding="max_length")

tokenized_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

print(f"Size of tokenized data: {len(tokenized_dataset)} examples")

Map:   0%|          | 0/63270 [00:00<?, ? examples/s]

Size of tokenized data: 63270 examples


## Train!

We select parameters close to LlamaTurk:
- `learning_rate = 3e-4`
- `warmup_steps = 500`
- `weight_decay = 0.01`

Use checkpoints to overcome kaggle limit - save them to wandb

In [17]:
BATCH_SIZE = 4
GRAD_ACCUM = 4                      # effective batch = 16
LEARNING_RATE = 3e-4
MAX_STEPS = 3000                   # mb more?
SAVE_STEPS = 500                   # save each ... steps
LOG_STEPS = 500
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.01

In [18]:
training_args = TrainingArguments(
    output_dir="./continual_checkpoints",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    logging_steps=LOG_STEPS,
    save_steps=SAVE_STEPS,
    report_to=["wandb"],
    run_name=WANDB_RUN_NAME,
    remove_unused_columns=False,
    fp16=True,
    max_steps=MAX_STEPS,
    save_total_limit=3,              # store locally
    load_best_model_at_end=False, # no validation, since it is literrally learning new lang
    disable_tqdm=False,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [19]:
# helper to work with checpoints

def save_checkpoint_to_wandb(local_checkpoint_dir, step):
    artifact_name = f"{CHECKPOINT_ARTIFACT_PREFIX}_step{step}"
    artifact = wandb.Artifact(name=artifact_name, type="checkpoint",
                              description=f"Checkpoint at step {step} for experiment {EXPERIMENT}")
    artifact.add_dir(local_checkpoint_dir)
    wandb.log_artifact(artifact)
    print(f"Checkpoint {step} saved as artifact {artifact_name}")

class WandbCheckpointCallback(TrainerCallback):
    def __init__(self, trainer):
        self.trainer = trainer
    def on_save(self, args, state, control, **kwargs):
        if state.is_world_process_zero:
            save_dir = args.output_dir
            checkpoints = [d for d in os.listdir(save_dir) if d.startswith("checkpoint-")]
            if checkpoints:
                last_checkpoint = sorted(checkpoints, key=lambda x: int(x.split("-")[-1]))[-1]
                local_path = os.path.join(save_dir, last_checkpoint)
                save_checkpoint_to_wandb(local_path, state.global_step)

In [20]:
wandb.init(project=WANDB_PROJECT, name=WANDB_RUN_NAME)

# manual continuation
# resume_checkpoint = None
checkpoint_artifact = wandb.use_artifact("e278979-metu-middle-east-technical-university/bashllama-continual-training-B/continual_B_checkpoint_step1000:latest", 
                                         type="checkpoint")
checkpoint_dir = checkpoint_artifact.download()
resume_checkpoint = checkpoint_dir

wandb: Finishing previous runs because reinit is set to 'default'.
wandb: updating run metadata
wandb: 🚀 View run astral-mountain-14 at: https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-embedding-alignment/runs/cr0bcypr
wandb: ⭐️ View project at: https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-embedding-alignment
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260528_170323-cr0bcypr/logs
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260528_170428-kll7tnb1
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run continual_B
wandb: ⭐️ View project at https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-continual-training-B
wandb: 🚀 View run at https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-continual-training-B/runs/kll7tnb1
wandb: Downloading large ar

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

In [22]:
from transformers.integrations import WandbCallback
trainer.add_callback(WandbCallback())

You are adding a <class 'transformers.integrations.integration_utils.WandbCallback'> to the callbacks of this Trainer, but there is already one. The currentlist of callbacks is
:DefaultFlowCallback
WandbCallback
NotebookProgressCallback


In [23]:
from transformers import TrainerCallback

class ConsoleLogCallback(TrainerCallback):
    def __init__(self, log_steps=50):
        self.log_steps = log_steps

    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.global_step % self.log_steps == 0 and logs:
            loss = logs.get('loss', None)
            lr = logs.get('learning_rate', None)
            print(f"Step {state.global_step}: loss = {loss}, lr = {lr}")

trainer.add_callback(ConsoleLogCallback(log_steps=50))

In [24]:
callback = WandbCheckpointCallback(trainer)
trainer.add_callback(callback)

In [25]:
trainer.train(resume_from_checkpoint=resume_checkpoint)

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such a

Step,Training Loss
1500,1.525400
2000,1.420373
2500,1.361702
3000,1.318188


Step 1500: loss = 1.5254002685546875, lr = 0.00018012


wandb: Adding directory to artifact (continual_checkpoints/checkpoint-1500)... Done. 0.2s
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Checkpoint 1500 saved as artifact continual_B_checkpoint_step1500
Step 2000: loss = 1.420372802734375, lr = 0.00012011999999999998


...

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step 3000: loss = 1.3181884765625, lr = 1.2e-07


wandb: Adding directory to artifact (continual_checkpoints/checkpoint-3000)... Done. 0.1s


Checkpoint 3000 saved as artifact continual_B_checkpoint_step3000
Step 3000: loss = None, lr = None


TrainOutput(global_step=3000, training_loss=0.9376105753580729, metrics={'train_runtime': 32698.9766, 'train_samples_per_second': 1.468, 'train_steps_per_second': 0.092, 'total_flos': 4.96025638797312e+17, 'train_loss': 0.9376105753580729, 'epoch': 0.7586294095334429})

In [26]:
final_dir = f"/kaggle/working/continual_{EXPERIMENT}_final"
model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)

artifact = wandb.Artifact(
    name=f"llama2_bashkir_continual_{EXPERIMENT}",
    type="model",
    description=f"Continual training on Bashkir corpus, experiment {EXPERIMENT}, steps {trainer.state.global_step}"
)
artifact.add_dir(final_dir)
wandb.log_artifact(artifact)
print(f"Final model saved to {final_dir} and as artifact 'llama2_bashkir_continual_{EXPERIMENT}'")

wandb.finish()

wandb: Adding directory to artifact (/kaggle/working/continual_B_final)... Done. 0.1s
wandb: uploading artifact continual_B_checkpoint_step3000; updating run metadata; uploading artifact llama2_bashkir_continual_B


Final model saved to /kaggle/working/continual_B_final and as artifact 'llama2_bashkir_continual_B'


wandb: uploading artifact continual_B_checkpoint_step3000; uploading artifact llama2_bashkir_continual_B
wandb: uploading config.yaml; uploading output.log; uploading wandb-summary.json
wandb: 
wandb: Run history:
wandb:         train/epoch ▁▁▃▃▆▆████
wandb:   train/global_step ▁▁▃▃▆▆████
wandb:     train/grad_norm ▂▂▁▁▄▄██
wandb: train/learning_rate ██▆▆▃▃▁▁
wandb:          train/loss ██▄▄▂▂▁▁
wandb: 
wandb: Run summary:
wandb:               total_flos 4.96025638797312e+17
wandb:              train/epoch 0.75863
wandb:        train/global_step 3000
wandb:          train/grad_norm 1.75051
wandb:      train/learning_rate 0.0
wandb:               train/loss 1.31819
wandb:               train_loss 0.93761
wandb:            train_runtime 32698.9766
wandb: train_samples_per_second 1.468
wandb:   train_steps_per_second 0.092
wandb: 
wandb: 🚀 View run continual_B at: https://wandb.ai/e278979-metu-middle-east-technical-university/bashllama-continual-training-B/runs/kll7tnb1
wandb: ⭐️ View proj

In [27]:
del model
del trainer
torch.cuda.empty_cache()